In [1]:
!pip install -q transformers peft datasets accelerate

In [2]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 20.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [6]:
import json
import torch

from google.colab import drive
drive.mount('/content/drive')

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, get_peft_model, TaskType

#load and preprocess Data
with open("/content/drive/My Drive/01_Now/01_NUS/ORBITAL/Milestone2/labelling_checkpoint_clean.json", "r", encoding="utf-8") as f:
    raw_data = json.load(f)

#shift labels from 1-5 to 0-4 and ensure no null/missing data is passed
processed_data = {
    "text": [item["explanation"] for item in raw_data.values() if "explanation" in item and "sentiment_score" in item and item["sentiment_score"] is not None],
    "label": [item["sentiment_score"] - 1 for item in raw_data.values() if "explanation" in item and "sentiment_score" in item and item["sentiment_score"] is not None]
}

dataset = Dataset.from_dict(processed_data)

#80/20 train-test split
dataset = dataset.train_test_split(test_size=0.2, seed=42)

#tokenizer and Base Model
model_id = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=512)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(
    model_id,
    num_labels=5
)

#low rank adaptation
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"] # Target attention layers
)

peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()


training_args = TrainingArguments(
    output_dir="./module_scorer_results",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch", # <-- Updated argument here
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=10
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    processing_class=tokenizer, # <-- Updated argument here
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer)
)

trainer.train()

# save lora adapters
peft_model.save_pretrained("./deberta-lora-module-scorer")
tokenizer.save_pretrained("./deberta-lora-module-scorer")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Map:   0%|          | 0/1520 [00:00<?, ? examples/s]

Map:   0%|          | 0/381 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier

trainable params: 298,757 || all params: 184,724,746 || trainable%: 0.1617


[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Epoch,Training Loss,Validation Loss
1,1.215161,1.203454
2,0.771717,0.648861
3,0.396962,0.378868
4,0.270308,0.358191
5,0.190936,0.359979
6,0.223983,0.240095
7,0.127381,0.213488
8,0.150577,0.193671
9,0.077488,0.199047
10,0.112416,0.188132


('./deberta-lora-module-scorer/tokenizer_config.json',
 './deberta-lora-module-scorer/tokenizer.json')